In [ ]:
# 🍓 Strawberry Leaf Disease Detection using CNN
# Google Colab Notebook
# Classes: Healthy and Leaf Scorch

import os
import zipfile
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Upload your strawberry dataset ZIP
# Expected structure:
# strawberry_dataset.zip
# ├── Healthy/
# └── Leaf_Scorch/

from google.colab import files

uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
print("Uploaded:", zip_file)

In [ ]:
# Extract dataset

extract_path = "/content/strawberry_dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
# Automatically find the folder containing class folders

def find_dataset_directory(base_path):
    for root, dirs, files in os.walk(base_path):
        image_folders = []
        for d in dirs:
            folder_path = os.path.join(root, d)
            try:
                images = [
                    f for f in os.listdir(folder_path)
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
                ]
                if images:
                    image_folders.append(d)
            except:
                pass
        if len(image_folders) >= 2:
            return root
    return base_path

DATASET_DIR = find_dataset_directory(extract_path)
print("Dataset directory:", DATASET_DIR)

In [ ]:
# Check classes and image counts

classes = [
    folder for folder in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, folder))
]
classes.sort()

print("Classes:", classes)
print("Number of classes:", len(classes))

total_images = 0
for class_name in classes:
    class_path = os.path.join(DATASET_DIR, class_name)
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]
    print(f"{class_name}: {len(images)} images")
    total_images += len(images)

print("Total images:", total_images)

In [ ]:
# Display sample images

plt.figure(figsize=(12, 8))
plot_index = 1

for class_name in classes:
    class_path = os.path.join(DATASET_DIR, class_name)
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]
    selected = random.sample(images, min(4, len(images)))

    for image_name in selected:
        image_path = os.path.join(class_path, image_name)
        image = tf.keras.utils.load_img(image_path, target_size=(224, 224))

        plt.subplot(len(classes), 4, plot_index)
        plt.imshow(image)
        plt.title(class_name)
        plt.axis("off")
        plot_index += 1

plt.tight_layout()
plt.show()

In [ ]:
# Create training and temporary validation/test datasets

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

validation_test_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

print("Datasets created!")

In [ ]:
# Split the 30% holdout approximately into validation and test

num_batches = tf.data.experimental.cardinality(validation_test_dataset).numpy()
test_batches = num_batches // 2

test_dataset = validation_test_dataset.take(test_batches)
validation_dataset = validation_test_dataset.skip(test_batches)

print("Train batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation batches:", tf.data.experimental.cardinality(validation_dataset).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_dataset).numpy())

In [ ]:
# Normalize images and optimize pipeline

normalization_layer = layers.Rescaling(1./255)

train_dataset = train_dataset.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
validation_dataset = validation_dataset.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
test_dataset = test_dataset.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(AUTOTUNE)

print("Preprocessing complete!")

In [ ]:
# Data augmentation

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.10),
], name="data_augmentation")

In [ ]:
# Build CNN model

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(len(classes), activation="softmax")
])

model.summary()

In [ ]:
# Compile model

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

In [ ]:
# Training callbacks

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/best_strawberry_cnn.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

In [ ]:
# Train CNN

EPOCHS = 20

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping, checkpoint]
)

In [ ]:
# Plot accuracy

plt.figure(figsize=(10, 6))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Strawberry Disease CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Plot loss

plt.figure(figsize=(10, 6))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Strawberry Disease CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate on test set

test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Generate predictions

y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Predictions generated!")

In [ ]:
# Classification report

print(classification_report(
    y_true,
    y_pred,
    target_names=classes,
    zero_division=0
))

In [ ]:
# Calculate accuracy, precision, recall and F1

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("================================")
print(" STRAWBERRY CNN PERFORMANCE")
print("================================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1 Score  : {f1 * 100:.2f}%")

In [ ]:
# Confusion matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Strawberry Disease Confusion Matrix")
plt.show()

In [ ]:
# Display sample test predictions

plt.figure(figsize=(15, 10))

for images, labels in test_dataset.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(min(12, len(images))):
        predicted_class = np.argmax(predictions[i])
        actual_class = labels[i].numpy()

        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(
            f"Actual: {classes[actual_class]}\n"
            f"Predicted: {classes[predicted_class]}"
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Save trained model

MODEL_PATH = "/content/strawberry_disease_cnn.keras"
model.save(MODEL_PATH)

print("Model saved:", MODEL_PATH)

In [ ]:
# Download trained model

from google.colab import files
files.download("/content/strawberry_disease_cnn.keras")

In [ ]:
# Upload a new strawberry leaf image for prediction

from google.colab import files

uploaded_test = files.upload()
test_image_path = list(uploaded_test.keys())[0]

print("Testing:", test_image_path)

In [ ]:
# Predict disease on a new image

image = tf.keras.utils.load_img(
    test_image_path,
    target_size=(224, 224)
)

image_array = tf.keras.utils.img_to_array(image)
image_array = image_array / 255.0
image_array = np.expand_dims(image_array, axis=0)

prediction = model.predict(image_array, verbose=0)[0]

predicted_index = np.argmax(prediction)
predicted_class = classes[predicted_index]
confidence = prediction[predicted_index] * 100

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"Prediction: {predicted_class}\n"
    f"Confidence: {confidence:.2f}%"
)
plt.show()

print("======================================")
print("🍓 STRAWBERRY DISEASE DETECTION")
print("======================================")
print("Result      :", predicted_class)
print(f"Confidence  : {confidence:.2f}%")
print("======================================")

In [ ]:
# Show probability for every class

print("Class Probabilities")
print("-------------------")

for i, class_name in enumerate(classes):
    print(f"{class_name}: {prediction[i] * 100:.2f}%")

In [ ]:
# Save performance report

report = classification_report(
    y_true,
    y_pred,
    target_names=classes,
    zero_division=0
)

with open("/content/strawberry_cnn_report.txt", "w") as f:
    f.write("STRAWBERRY DISEASE CNN MODEL\n")
    f.write("============================\n\n")
    f.write(f"Accuracy: {accuracy * 100:.2f}%\n")
    f.write(f"Precision: {precision * 100:.2f}%\n")
    f.write(f"Recall: {recall * 100:.2f}%\n")
    f.write(f"F1 Score: {f1 * 100:.2f}%\n\n")
    f.write("Classification Report\n")
    f.write(report)

print("Report saved: /content/strawberry_cnn_report.txt")